In [1]:
import duckdb

In [2]:
con = duckdb.connect("research.duckdb")

In [3]:
con.execute("SHOW TABLES").df()

,name
0,current_eps_features
1,current_eps_raw
2,historical_eps_features
3,historical_news_clean
4,historical_prices_clean
5,llm_data
6,master_developments
7,master_historical_eps
8,master_historical_prices
9,signal_price


### Get rid of news corresponding to iqids with no price data

In [4]:
con.execute("SELECT DISTINCT IQID FROM historical_prices_clean").df()

,IQID
0,7079247
1,4104164
2,4209952
3,133940468
4,6331068
...,...
3959,4208452
3960,4098794
3961,4660749
3962,102864


### Unnest the IQIDs list an create a row for each IQID in the news. Get rid of rows with no price data. Clean up the news table.

In [ ]:
def clean_news_data():
    con.execute("""
        CREATE OR REPLACE TABLE historical_news_clean AS
        WITH cleaned_news AS (
            -- Unnest and basic cast from your first step
            SELECT
                dev_date,
                CAST(UNNEST(IQIDS) AS INT) AS iqid,
                dev_type,
                headline,
                description
            FROM master_developments
        ),
        valid_iqids AS (
            -- Pre-filter the IDs to speed up the final join
            SELECT DISTINCT iqid FROM historical_prices_clean
        )

        SELECT 
            DATE(
                CASE 
                    WHEN cn.dev_date LIKE '%:%' THEN
                        STRPTIME(cn.dev_date, '%d/%m/%Y %H:%M:%S')
                    ELSE
                        STRPTIME(cn.dev_date, '%d/%m/%Y')
                END
            ) AS dev_date,
            cn.iqid,
            SPLIT_PART(u.company_name, ' (', 1) AS clean_name,
            REPLACE(
                NULLIF(
                    REGEXP_EXTRACT(u.company_name, '\\((?:.*:)?([A-Z\\. ]+)\\)', 1), ''), 
                ' ', '') AS ticker,        
            cn.dev_type,
            cn.headline,
            cn.description            

        FROM cleaned_news cn
        JOIN valid_iqids p 
            ON cn.iqid = p.iqid
        JOIN (
            SELECT iqid, ANY_VALUE(company_name) AS company_name
            FROM historical_eps_features
            GROUP BY iqid
        ) u
            ON cn.iqid = u.iqid
    """)

In [10]:
con.execute("""       
            CREATE OR REPLACE TABLE llm_data AS           
            SELECT
                dev_date,
                iqid,
                CONCAT(
                    clean_name,
                    ' (', ticker, '): ',
                    headline
                ) AS text_input
            FROM historical_news_clean
            WHERE dev_type NOT IN (
                'Company Conference Presentation',
                'Conference',
                'Earnings Release Date',
                'Earnings Call',
                'Earnings Release Date, Estimated',
                'Ex-Dividend Date, Regular',
                'Dividend Affirmation',
                'Annual General Meeting',
                'Shelf Registration Filing',
                'Shareholder or Analyst Call',
                'Special Call',
                'Proxy or Voting',
                'Analyst or Investor Day',
                'M&A Call',
                'Public Offering Lead Underwriter Change',
                'Client Announcement',
                'End of Lock-up Period',
                'Proposals',
                'Change in Company Bylaws or Rules'
            )          
            """)

In [13]:
con.execute("SHOW TABLES").df()

,name
0,current_eps_features
1,current_eps_raw
2,historical_eps_features
3,historical_news_clean
4,historical_prices_clean
5,llm_data
6,master_developments
7,master_historical_eps
8,master_historical_prices
9,signal_price


### Load the finbert model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

### Copy llm table to new db to use on Noteable

In [14]:
con.close()


In [15]:
con = duckdb.connect("research.duckdb")
con.execute("ATTACH 'llm_data.duckdb' AS target_db")
con.execute("CREATE TABLE target_db.llm_data AS SELECT * FROM main.llm_data")
con.execute("DETACH target_db")
con.close()

In [18]:
con = duckdb.connect("llm_data.duckdb")
con.execute("SELECT COUNT() FROM llm_data LIMIT 10").df()

,count_star()
0,786068


In [9]:
df.to_csv("test_file.csv")